In [ ]:
!pip install timm kaggle torch torchvision pillow opencv-python matplotlib


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [ ]:
from google.colab import files
files.upload()   # upload kaggle.json


Saving kaggle.json to kaggle (1).json


{'kaggle (1).json': b'{"username":"shaikgousepeer","key":"fe3b161519d97fde8a190c3428daf679"}'}

In [ ]:
!kaggle datasets download -d udaykarthik21bce9252/vitamin-defficiency-dataset
!unzip vitamin-defficiency-dataset.zip -d raw_dataset


Dataset URL: https://www.kaggle.com/datasets/udaykarthik21bce9252/vitamin-defficiency-dataset
License(s): apache-2.0


In [ ]:
FINAL_CLASSES = [
    "Vitamin_A",
    "Vitamin_B_Complex",
    "Vitamin_C",
    "Vitamin_D",
    "Vitamin_E",
    "Vitamin_K"
]


In [ ]:
import os, shutil

# ✅ Correct source path (IMPORTANT)
SRC = "raw_dataset/dataset"
DST = "clean_dataset"

os.makedirs(DST, exist_ok=True)

FINAL_CLASSES = [
    "Vitamin_A",
    "Vitamin_B_Complex",
    "Vitamin_C",
    "Vitamin_D",
    "Vitamin_E",
    "Vitamin_K"
]

def map_class(folder_name):
    name = folder_name.lower()
    if "vitamin a" in name:
        return "Vitamin_A"
    if "vitamin b" in name:
        return "Vitamin_B_Complex"
    if "vitamin c" in name:
        return "Vitamin_C"
    if "vitamin d" in name:
        return "Vitamin_D"
    if "vitamin e" in name:
        return "Vitamin_E"
    if "vitamin k" in name:
        return "Vitamin_K"
    return None  # noisy class (zinc/iron/etc)

counts = {c: 0 for c in FINAL_CLASSES}

for folder in os.listdir(SRC):
    src_folder = os.path.join(SRC, folder)
    if not os.path.isdir(src_folder):
        continue

    target_class = map_class(folder)
    if target_class is None:
        continue  # skip noisy folders

    dst_folder = os.path.join(DST, target_class)
    os.makedirs(dst_folder, exist_ok=True)

    for img in os.listdir(src_folder):
        if img.lower().endswith(('.jpg', '.jpeg', '.png')):
            shutil.copy(
                os.path.join(src_folder, img),
                os.path.join(dst_folder, f"{folder}_{img}")
            )
            counts[target_class] += 1

print("✅ CLEAN DATASET CREATED")
for k, v in counts.items():
    print(f"{k}: {v} images")


In [ ]:
import os

print("\nVerification:\n")
for cls in os.listdir("clean_dataset"):
    cls_path = os.path.join("clean_dataset", cls)
    imgs = [f for f in os.listdir(cls_path)
            if f.lower().endswith(('.jpg','.jpeg','.png'))]
    print(cls, "→", len(imgs), "images")


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

transform = transforms.Compose([
    transforms.Resize((380,380)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

dataset = datasets.ImageFolder(DST, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False)

CLASS_NAMES = dataset.classes


efficientnet b4 model download

In [ ]:
import torch, timm, torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = timm.create_model(
    "efficientnet_b4",
    pretrained=True,
    num_classes=len(CLASS_NAMES)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


traing


In [ ]:
def train_epoch(loader):
    model.train()
    correct, total = 0, 0
    for x,y in loader:
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out,y)
        loss.backward()
        optimizer.step()
        correct += (out.argmax(1)==y).sum().item()
        total += y.size(0)
    return correct/total

def eval_epoch(loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            out = model(x)
            correct += (out.argmax(1)==y).sum().item()
            total += y.size(0)
    return correct/total

for e in range(5):
    print("Train:", train_epoch(train_loader),
          "Val:", eval_epoch(val_loader))


In [ ]:
PHASE1_PATH = "/content/drive/MyDrive/vitamin_phase1_efficientnet_b4.pth"

torch.save(model.state_dict(), PHASE1_PATH)

print("✅ Phase-1 model saved to Google Drive")
